
# Chapter 21 — Log File:



A log file is basically the application's diary.
It records what happened while your application was running, especially things that you may need to investigate later.

In [33]:
# Will make a log file in JSON format, which will keep data in this kind of format

In [34]:
# {
#   "timestamp": "2026-10-06T22:30:01",
#   "level": "INFO",
#   "event": "LLM_REQUEST",

#   "model": "openai/gpt-oss-20b",

#   "attempt": 1,

#   "status": "SUCCESS",

#   "input_tokens": 102,
#   "output_tokens": 1308,
#   "total_tokens": 1410,
#   "reasoning_tokens": 836,

#   "latency_seconds": 2.41,

#   "error": null
# }

      LLM Application
                    ↓
              Try Request
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
       SUCCESS              ERROR
          ↓                   ↓
     Get metadata          Log error
          ↓                   ↓
     Token details         Retry
          ↓                   ↓
          └─────────┬─────────┘
                    ↓
              JSONL Log
                    ↓
          Request + Tokens
          + Latency + Error

# We are moving API key to .env file ...create .env file and Store your API 

GROQ_API_KEY=YOUR_API_KEY 

This will not expose your API 

In [ ]:
from dotenv import load_dotenv
import os 

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("API key Not found")
print("GROQ_API_KEY is configured")

In [49]:
# %pip install -U langchain langchain-core langchain-groq

In [50]:
#!pip install -U langchain-openai

In [51]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

In [52]:
# Creating a JSON file name with date when the code we run

In [53]:
from datetime import datetime

file_name = f"llm_app_{datetime.now().strftime('%Y-%m-%d')}.jsonl"

print(file_name)

llm_app_2026-10-07.jsonl


In [54]:
import json
from datetime import datetime

In [55]:
import json
from datetime import datetime


def write_log(
    level,
    event,
    model,
    attempt,
    status,
    input_tokens=None,
    output_tokens=None,
    reasoning_tokens=None,
    total_tokens=None,
    latency_seconds=None,
    error=None
):

    log_data = {
        "timestamp": datetime.now().isoformat(),
        "level": level,
        "event": event,
        "model": model,
        "attempt": attempt,
        "status": status,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": total_tokens,
        "latency_seconds": latency_seconds,
        "error": error
    }

    with open(file_name, "a") as file:
        file.write(json.dumps(log_data) + "\n")

In [56]:
write_log(
    level="INFO",
    event="TEST",
    model="openai/gpt-oss-20b",
    attempt=1,
    status="SUCCESS"
)

In [57]:
import time


def ask_llm(prompt):

    start_time = time.time()

    try:

        response = llm.invoke(prompt)

        latency = time.time() - start_time

        usage = response.usage_metadata

        write_log(
            level="INFO",
            event="LLM_REQUEST",
            model="openai/gpt-oss-20b",
            attempt=1,
            status="SUCCESS",
            input_tokens=usage.get("input_tokens"),
            output_tokens=usage.get("output_tokens"),
            reasoning_tokens=usage.get(
                "output_token_details", {}
            ).get("reasoning"),
            total_tokens=usage.get("total_tokens"),
            latency_seconds=round(latency, 3)
        )

        return response.content

    except Exception as e:

        latency = time.time() - start_time

        write_log(
            level="ERROR",
            event="LLM_REQUEST",
            model="openai/gpt-oss-20b",
            attempt=1,
            status="FAILED",
            latency_seconds=round(latency, 3),
            error=str(e)
        )

        return None

In [58]:
answer = ask_llm(
    "Explain RAG in simple words."
)

print(answer)

**RAG = Retrieval‑Augmented Generation**

Think of a language model (like ChatGPT) as a brilliant student who can *generate* sentences but only from what it “learned” during training.  
If you ask it a question about something that happened after its training cutoff, or something very niche, it might not know the answer.

RAG adds a *search* step so the model can look up fresh facts, then uses those facts to answer more accurately.

**How it works in three simple steps**

| Step | What happens | Analogy |
|------|--------------|---------|
| **Retrieve** | The system looks for relevant documents or passages from a big database (e.g., Wikipedia, a corporate knowledge base). | A librarian quickly pulls out the exact book or page you need. |
| **Augment** | The retrieved text is fed into the language model as extra context. | You hand the librarian’s copy to the student so they can study it. |
| **Generate** | The model writes an answer that blends its own language skills with the new info